In [ ]:
%%bash
set -euo pipefail

echo '════════════════════════════════════════════'
echo ' CELL 1 — System tools + Install Ollama'
echo '════════════════════════════════════════════'

apt-get update -qq
apt-get install -y zstd pciutils curl 2>&1 | tail -3
echo '[OK] apt packages installed'

npm install -g localtunnel 2>&1 | tail -2
echo "[OK] localtunnel $(lt --version)"

echo ''
echo '── Installing Ollama ──'
curl -fsSL https://ollama.com/install.sh | sh
ollama --version

echo ''
echo '[DONE] Cell 1 complete'

════════════════════════════════════════════
 CELL 1 — System tools + Install Ollama
════════════════════════════════════════════

/sbin/ldconfig.real: /usr/local/lib/libur_adapter_level_zero.so.0 is not a symbolic link

[OK] apt packages installed
npm notice To update run: npm install -g npm@11.13.0
npm notice
[OK] localtunnel 2.0.2

── Installing Ollama ──

[DONE] Cell 1 complete


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> NVIDIA GPU installed.
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [ ]:
import subprocess, os, glob, time, requests

print("════════════════════════════════════════════")
print(" CELL 2 — GPU Fix + Ollama GPU Start")
print("════════════════════════════════════════════")

subprocess.run(["pkill", "-f", "ollama"], capture_output=True)
time.sleep(2)
print("[OK] Killed any stale Ollama process")

print("\n── Discovering NVIDIA/CUDA libs ──")
r = subprocess.run("find /usr/lib64-nvidia /usr/local/cuda* -name 'libcuda.so*' 2>/dev/null",
                   shell=True, capture_output=True, text=True)
print(r.stdout.strip() or "WARNING: No libcuda found!")

ollama_lib = "/usr/local/lib/ollama"
print(f"\n── Ollama lib dir: {ollama_lib} ──")
r2 = subprocess.run(f"ls {ollama_lib}/", shell=True, capture_output=True, text=True)
print(r2.stdout.strip())

nvidia_dir = "/usr/lib64-nvidia"
cuda_dir   = "/usr/local/cuda/lib64"
cuda128    = "/usr/local/cuda-12.8/targets/x86_64-linux/lib"

needed = {
    "libcuda.so":         [f"{nvidia_dir}/libcuda.so",      f"{nvidia_dir}/libcuda.so.1"],
    "libcuda.so.1":       [f"{nvidia_dir}/libcuda.so.1"],
    "libnvidia-ml.so.1":  [f"{nvidia_dir}/libnvidia-ml.so.1"],
    "libnvidia-ml.so":    [f"{nvidia_dir}/libnvidia-ml.so.1"],
    "libcudart.so.12":    [f"{cuda_dir}/libcudart.so.12",   f"{cuda128}/libcudart.so.12"],
}

print("\n── Creating symlinks ──")
for dst_name, srcs in needed.items():
    dst = os.path.join(ollama_lib, dst_name)
    if os.path.lexists(dst):
        os.remove(dst)
    for src in srcs:
        if os.path.exists(src):
            os.symlink(src, dst)
            print(f"  ✅ {src} → {dst}")
            break
    else:
        print(f"  ⚠️  MISSING: {dst_name} (no source found)")

gpu_env = os.environ.copy()
gpu_env.pop("CUDA_VISIBLE_DEVICES", None)
gpu_env["LD_LIBRARY_PATH"] = (
    f"{nvidia_dir}:{ollama_lib}:/usr/local/cuda/lib64:"
    + os.environ.get("LD_LIBRARY_PATH", "")
)
gpu_env["OLLAMA_KEEP_ALIVE"] = "2h"
gpu_env["OLLAMA_DEBUG"]      = "INFO"

print(f"\n── Starting Ollama daemon ──")
print(f"LD_LIBRARY_PATH starts with: {gpu_env['LD_LIBRARY_PATH'][:80]}...")
log = open("/content/ollama.log", "w")
proc = subprocess.Popen(["ollama", "serve"], env=gpu_env, stdout=log, stderr=log,
                        preexec_fn=os.setpgrp)
print(f"[OK] Ollama PID: {proc.pid}")

for i in range(25):
    try:
        requests.get("http://localhost:11434/api/tags", timeout=2)
        print(f"[OK] Ollama API responsive after {i+1}s")
        break
    except:
        time.sleep(1)

time.sleep(2)

print("\n── Ollama startup log (GPU detection lines) ──")
log_txt = open("/content/ollama.log").read()
for line in log_txt.splitlines():
    kws = ["gpu", "cuda", "vram", "nvidia", "error", "warn", "discovered", "total_vram"]
    if any(k in line.lower() for k in kws):
        print(line)

print("\n── Checking model ──")
tags = requests.get("http://localhost:11434/api/tags").json()
models = [m["name"] for m in tags.get("models", [])]
print(f"Installed: {models}")
if not any("llama3.1:8b" in m for m in models):
    print("Pulling llama3.1:8b ...")
    subprocess.run(["ollama", "pull", "llama3.1:8b"], check=True)
    print("[OK] Model pulled")

print("\n── Warming up model (60-90s first load) ──")
t0 = time.time()
r3 = requests.post("http://localhost:11434/api/chat", json={
    "model": "llama3.1:8b",
    "messages": [{"role": "user", "content": "Reply with the word READY only."}],
    "stream": False,
    "options": {"temperature": 0, "num_predict": 5}
}, timeout=(10, 300))
elapsed = time.time() - t0
d = r3.json()
load_s = d.get("load_duration", 0) / 1e9
eval_s = d.get("eval_duration", 0) / 1e9
reply  = d["message"]["content"].strip()
print(f"Reply   : {reply}")
print(f"Total   : {elapsed:.1f}s  |  Load: {load_s:.1f}s  |  Eval: {eval_s:.1f}s")
if eval_s > 0:
    tps = d.get("eval_count", 0) / eval_s
    print(f"Speed   : {tps:.1f} tok/s  {'← GPU (>30 t/s)' if tps > 30 else '← CPU (<10 t/s typical)'}")

print("\n── ollama ps ──")
ps = subprocess.run(["ollama", "ps"], capture_output=True, text=True)
print(ps.stdout)

print("── nvidia-smi ──")
smi = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.used,memory.total", "--format=csv,noheader"],
    capture_output=True, text=True)
print(smi.stdout)

used_mib = int(smi.stdout.split(",")[1].strip().split()[0]) if smi.stdout else 0
if used_mib > 4000:
    print(f"✅ GPU confirmed — {used_mib} MiB used")
elif "GPU" in ps.stdout:
    print(f"✅ ollama ps shows GPU")
else:
    print(f"⚠️  Still on CPU ({used_mib} MiB). Check log lines above for CUDA errors.")

print("\n[DONE] Cell 2 complete")

════════════════════════════════════════════
 CELL 2 — GPU Fix + Ollama GPU Start
════════════════════════════════════════════
[OK] Killed any stale Ollama process

── Discovering NVIDIA/CUDA libs ──
/usr/lib64-nvidia/libcuda.so.1
/usr/lib64-nvidia/libcuda.so.580.82.07
/usr/lib64-nvidia/libcuda.so
/usr/local/cuda-12.8/compat/libcuda.so.570.124.06
/usr/local/cuda-12.8/compat/libcuda.so.1
/usr/local/cuda-12.8/compat/libcuda.so
/usr/local/cuda-12.8/targets/x86_64-linux/lib/stubs/libcuda.so

── Ollama lib dir: /usr/local/lib/ollama ──
cuda_v12
cuda_v13
include
libggml-base.so
libggml-base.so.0
libggml-base.so.0.0.0
libggml-cpu-alderlake.so
libggml-cpu-haswell.so
libggml-cpu-icelake.so
libggml-cpu-sandybridge.so
libggml-cpu-skylakex.so
libggml-cpu-sse42.so
libggml-cpu-x64.so
mlx_cuda_v13
vulkan

── Creating symlinks ──
  ✅ /usr/lib64-nvidia/libcuda.so → /usr/local/lib/ollama/libcuda.so
  ✅ /usr/lib64-nvidia/libcuda.so.1 → /usr/local/lib/ollama/libcuda.so.1
  ✅ /usr/lib64-nvidia/libnvidia-ml

In [ ]:
%%bash
set -euo pipefail
echo '════════════════════════════════════════════'
echo ' CELL 3 — Clone + Patch'
echo '════════════════════════════════════════════'
cd /content
rm -rf medintake-ai
git clone https://github.com/priyansh-saxena1/medintake-ai.git
cd medintake-ai
echo "[OK] commit: $(git rev-parse --short HEAD)"
pip install -r requirements.txt 2>&1 | tail -4
echo '[OK] pip done'

python3 << 'PYEOF'
import pathlib
p   = pathlib.Path('/content/medintake-ai/app/llm.py')
src = p.read_text()
changed = False

patches = [
    ("PATCH 1 timeout",
     'requests.post(self.api_url, json=payload, timeout=60)',
     'requests.post(self.api_url, json=payload, timeout=(10, 300))'),
    ("PATCH 2 OLLAMA_HOST",
     'self.api_url = "http://localhost:11434/api/chat"',
     'self.api_url = os.environ.get("OLLAMA_HOST","http://localhost:11434") + "/api/chat"'),
    ("PATCH 3 MODEL_NAME default",
     'self.model_name = os.environ.get("MODEL_NAME", "qwen2.5:0.5b")',
     'self.model_name = os.environ.get("MODEL_NAME", "llama3.1:8b")'),
    ("PATCH 4 response key",
     'raw = data.get("response", "")',
     'raw = data.get("message", {}).get("content", "")'),
]

for name, old, new in patches:
    if old in src:
        src = src.replace(old, new, 1)
        changed = True
        print(f"[APPLIED] {name}")
    elif new in src:
        print(f"[SKIP]    {name}")
    else:
        print(f"[WARN]    {name} target not found")

if changed:
    p.write_text(src)
    print("[OK] llm.py saved")

import py_compile
py_compile.compile(str(p), doraise=True)
print("[OK] syntax valid")
PYEOF

echo ''
echo '── Tests (MockLLM) ──'
cd /content/medintake-ai
MOCK_LLM=true python3 -m pytest tests/ -v --tb=short 2>&1
echo '[DONE] Cell 3'


════════════════════════════════════════════
 CELL 3 — Clone + Patch
════════════════════════════════════════════
[OK] commit: eb1b955
[OK] pip done
[APPLIED] PATCH 1 timeout
[APPLIED] PATCH 2 OLLAMA_HOST
[APPLIED] PATCH 3 MODEL_NAME default
[SKIP]    PATCH 4 response key
[OK] llm.py saved
[OK] syntax valid

── Tests (MockLLM) ──
============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/medintake-ai
configfile: pytest.ini
plugins: asyncio-1.3.0, langsmith-0.7.30, typeguard-4.5.1, anyio-4.13.0
asyncio: mode=Mode.AUTO, debug=False, asyncio_default_fixture_loop_scope=function, asyncio_default_test_loop_scope=function
collecting ... collected 11 items

tests/test_e2e.py::test_mock_llm_combined_call_basic_extraction PASSED   [  9%]
tests/test_e2e.py::test_mock_llm_emergency_detection PASSED              [ 18%]
tests/test_e2e.py::test_mock_llm_does

Cloning into 'medintake-ai'...


In [ ]:
# Auto-heal Ollama before each turn
import requests, subprocess, os, time
def ensure_ollama():
    try:
        requests.get("http://localhost:11434/api/tags", timeout=2); return
    except: pass
    print("⚠️ Ollama dead — restarting...")
    env = os.environ.copy()
    env.pop("CUDA_VISIBLE_DEVICES", None)
    env["LD_LIBRARY_PATH"] = "/usr/lib64-nvidia:/usr/local/lib/ollama:/usr/local/cuda/lib64"
    env["OLLAMA_KEEP_ALIVE"] = "2h"
    subprocess.Popen(["ollama","serve"], env=env,
                     stdout=open("/content/ollama.log","a"),
                     stderr=subprocess.STDOUT, preexec_fn=os.setpgrp)
    for _ in range(30):
        try: requests.get("http://localhost:11434/api/tags",timeout=2); print("✅ Ollama back"); return
        except: time.sleep(1)

ensure_ollama()

import sys, os, json, time, subprocess, requests
sys.path.insert(0, "/content/medintake-ai")
os.environ["MOCK_LLM"]    = "false"
os.environ["MODEL_NAME"]  = "llama3.1:8b"
os.environ["OLLAMA_HOST"] = "http://localhost:11434"

print("════════════════════════════════════════════")
print(" CELL 4 — Deep LLM Dataflow Debug")
print("════════════════════════════════════════════")

# A: Hardware
print("\n── A: Hardware status ──")
ps  = subprocess.run(["ollama","ps"], capture_output=True, text=True)
smi = subprocess.run(["nvidia-smi","--query-gpu=name,memory.used,memory.total",
                      "--format=csv,noheader"], capture_output=True, text=True)
print("ollama ps :", ps.stdout.strip())
print("nvidia-smi:", smi.stdout.strip())

# B: App env + object init
print("\n── B: App LLM object ──")
from app.llm import OllamaLLM, CombinedOutput, COMBINED_SYSTEM_PROMPT
llm = OllamaLLM()
print(f"model_name : {llm.model_name}")
print(f"api_url    : {llm.api_url}")

# C: Prompt construction
print("\n── C: System prompt ──")
print(COMBINED_SYSTEM_PROMPT)

transcript  = "Patient: I have chest pain"
currentjson = CombinedOutput().model_dump_json()
prompt = (
    f"CURRENT CLINICAL STATE:\n{currentjson}\n\n"
    f"FULL CONVERSATION TRANSCRIPT:\n{transcript}\n\n"
    "Instructions: Extract all new clinical facts, merge into state, "
    "generate ONE empathetic follow-up question. Return ONLY JSON."
)
print("\n── D: User prompt ──")
print(prompt)

# D: Raw HTTP
print("\n── E: Raw Ollama HTTP call ──")
payload = {
    "model": llm.model_name,
    "messages": [
        {"role":"system","content": COMBINED_SYSTEM_PROMPT},
        {"role":"user",  "content": prompt}
    ],
    "format": "json",
    "stream": False,
    "options": {"temperature": 0.0, "num_predict": 300}
}

t0   = time.time()
resp = requests.post(llm.api_url, json=payload, timeout=(10,300))
elapsed = time.time() - t0
full = resp.json()

load_s = full.get("load_duration",0) / 1e9
eval_s = full.get("eval_duration",1) / 1e9
tps    = full.get("eval_count",0) / eval_s

print(f"HTTP status   : {resp.status_code}")
print(f"Total time    : {elapsed:.2f}s")
print(f"Load duration : {load_s:.2f}s   {'GPU (fast)' if load_s < 1 else 'CPU (slow)'}")
print(f"Tokens/sec    : {tps:.1f}        {'GPU (>30)' if tps > 30 else 'CPU (<15)'}")
raw = full.get("message",{}).get("content","").strip()
print(f"\nRaw content:\n{raw}")

# E: Parsing
print("\n── F: JSON parse ──")
try:
    parsed = json.loads(raw)
    print("json.loads() OK")
    REQUIRED = {"chief_complaint","onset","location","duration",
                "character","severity","aggravating","relieving","ros","reply"}
    missing_k = REQUIRED - set(parsed.keys())
    extra_k   = set(parsed.keys()) - REQUIRED
    print(f"Missing keys : {missing_k or 'none'}")
    print(f"Extra keys   : {extra_k or 'none'}")
    print(json.dumps(parsed, indent=2))
except Exception as e:
    print(f"FAILED: {e}")

# F: Full pipeline
print("\n── G: Full app pipeline ──")
result = llm.combined_call(transcript, currentjson)
print("CombinedOutput:")
print(json.dumps(result.model_dump(), indent=2))

from app.graph import compute_stage, missing_from
stage   = compute_stage(result)
missing = missing_from(result)
print(f"\nStage   : {stage}")
print(f"Missing : {missing}")
print(f"Reply   : '{result.reply}'")

FALLBACK = {"", "Could you tell me more?", "Could you please repeat that?"}
if result.reply in FALLBACK:
    print("\nWARNING: FALLBACK REPLY — LLM output failed silently!")
    print("Check logs below:")
    print(open("/content/ollama.log").read()[-2000:])
else:
    print("\nOK: Real LLM reply returned")

print("[DONE] Cell 4")


════════════════════════════════════════════
 CELL 4 — Deep LLM Dataflow Debug
════════════════════════════════════════════

── A: Hardware status ──
ollama ps : NAME           ID              SIZE      PROCESSOR    CONTEXT    UNTIL            
llama3.1:8b    46e0c10c039e    5.5 GB    100% GPU     4096       2 hours from now
nvidia-smi: Tesla T4, 5367 MiB, 15360 MiB

── B: App LLM object ──
model_name : llama3.1:8b
api_url    : http://localhost:11434/api/chat

── C: System prompt ──
You are a clinical intake assistant AI. You have two jobs per turn:

JOB 1 (EXTRACT): Read the FULL conversation and update the clinical JSON state with any new information the patient provided. Only extract facts explicitly stated.

JOB 2 (RESPOND): Based on what is STILL MISSING from the clinical state, ask the patient ONE natural, empathetic question. Do NOT ask about things already filled in.

CRITICAL RULES:
- Output ONLY valid JSON, nothing else.
- Do NOT diagnose or give medical advice.
- Do NOT ask 

In [ ]:
# Auto-heal Ollama before each turn
import requests, subprocess, os, time
def ensure_ollama():
    try:
        requests.get("http://localhost:11434/api/tags", timeout=2); return
    except: pass
    print("⚠️ Ollama dead — restarting...")
    env = os.environ.copy()
    env.pop("CUDA_VISIBLE_DEVICES", None)
    env["LD_LIBRARY_PATH"] = "/usr/lib64-nvidia:/usr/local/lib/ollama:/usr/local/cuda/lib64"
    env["OLLAMA_KEEP_ALIVE"] = "2h"
    subprocess.Popen(["ollama","serve"], env=env,
                     stdout=open("/content/ollama.log","a"),
                     stderr=subprocess.STDOUT, preexec_fn=os.setpgrp)
    for _ in range(30):
        try: requests.get("http://localhost:11434/api/tags",timeout=2); print("✅ Ollama back"); return
        except: time.sleep(1)

ensure_ollama()

import subprocess, time, requests, os

subprocess.run(["pkill", "-f", "uvicorn"], capture_output=True)
time.sleep(2)

env = os.environ.copy()
env["MOCK_LLM"]    = "false"
env["MODEL_NAME"]  = "llama3.1:8b"
env["OLLAMA_HOST"] = "http://localhost:11434"

log  = open("/content/api.log", "w")
proc = subprocess.Popen(
    ["python", "-m", "uvicorn", "app.main:app",
     "--host", "0.0.0.0", "--port", "7860", "--log-level", "info"],
    cwd="/content/medintake-ai",
    env=env, stdout=log, stderr=log,
    preexec_fn=os.setpgrp
)
print(f"uvicorn PID: {proc.pid}")

for i in range(20):
    try:
        r = requests.get("http://localhost:7860/health", timeout=2)
        if r.status_code == 200:
            d = r.json()
            print(f"✅ FastAPI ready after {i+1}s")
            print(f"   mock_mode = {d.get('mock_mode')}  ← must be False")
            break
    except: pass
    print(f"  ...{i+1}s")
    time.sleep(1)
else:
    print("❌ Failed — dumping api.log:")
    print(open("/content/api.log").read()[-2000:])

uvicorn PID: 19612
  ...1s
  ...2s
✅ FastAPI ready after 3s
   mock_mode = False  ← must be False


In [ ]:
# Auto-heal Ollama before each turn
import requests, subprocess, os, time
def ensure_ollama():
    try:
        requests.get("http://localhost:11434/api/tags", timeout=2); return
    except: pass
    print("⚠️ Ollama dead — restarting...")
    env = os.environ.copy()
    env.pop("CUDA_VISIBLE_DEVICES", None)
    env["LD_LIBRARY_PATH"] = "/usr/lib64-nvidia:/usr/local/lib/ollama:/usr/local/cuda/lib64"
    env["OLLAMA_KEEP_ALIVE"] = "2h"
    subprocess.Popen(["ollama","serve"], env=env,
                     stdout=open("/content/ollama.log","a"),
                     stderr=subprocess.STDOUT, preexec_fn=os.setpgrp)
    for _ in range(30):
        try: requests.get("http://localhost:11434/api/tags",timeout=2); print("✅ Ollama back"); return
        except: time.sleep(1)

ensure_ollama()

import subprocess, time, requests

PUBLIC_IP = requests.get("https://ipv4.icanhazip.com", timeout=5).text.strip()
print(f"Tunnel password: {PUBLIC_IP}")
print("Starting tunnel...")

# Start lt as background process, capture output to file
tunnel_log = open("/content/tunnel.log", "w")
proc = subprocess.Popen(
    ["lt", "--port", "7860"],
    stdout=tunnel_log, stderr=tunnel_log,
    preexec_fn=__import__("os").setpgrp
)
print(f"Tunnel PID: {proc.pid}")

# Wait for URL to appear in log
for i in range(15):
    time.sleep(1)
    try:
        txt = open("/content/tunnel.log").read()
        if "loca.lt" in txt or "https://" in txt:
            for line in txt.splitlines():
                if "https://" in line:
                    print(f"\n🌐 PUBLIC URL: {line.strip()}")
            break
    except: pass
    print(f"  ...waiting for URL {i+1}s")
else:
    print("⚠️  URL not found yet — run: !cat /content/tunnel.log")

print("\n✅ Cell 5B done — proceed to Cell 6")

Tunnel password: 34.87.70.249
Starting tunnel...
Tunnel PID: 19630
  ...waiting for URL 1s

🌐 PUBLIC URL: your url is: https://proud-bears-drum.loca.lt

✅ Cell 5B done — proceed to Cell 6


In [ ]:

# Auto-heal Ollama before each turn
import requests, subprocess, os, time
def ensure_ollama():
    try:
        requests.get("http://localhost:11434/api/tags", timeout=2); return
    except: pass
    print("⚠️ Ollama dead — restarting...")
    env = os.environ.copy()
    env.pop("CUDA_VISIBLE_DEVICES", None)
    env["LD_LIBRARY_PATH"] = "/usr/lib64-nvidia:/usr/local/lib/ollama:/usr/local/cuda/lib64"
    env["OLLAMA_KEEP_ALIVE"] = "2h"
    subprocess.Popen(["ollama","serve"], env=env,
                     stdout=open("/content/ollama.log","a"),
                     stderr=subprocess.STDOUT, preexec_fn=os.setpgrp)
    for _ in range(30):
        try: requests.get("http://localhost:11434/api/tags",timeout=2); print("✅ Ollama back"); return
        except: time.sleep(1)

ensure_ollama()

import requests, json, time, subprocess

SESSION_ID = "debug_session_001"   # keep same across all turns
USER_MSG   = "I have chest pain"   # ← change each turn

print(f"Session: {SESSION_ID}  |  Message: {USER_MSG}")

t0 = time.time()
r  = requests.post("http://localhost:7860/chat",
    json={"session_id": SESSION_ID, "message": USER_MSG},
    timeout=120)
elapsed = time.time() - t0

d = r.json()
print(f"\nHTTP {r.status_code}  ({elapsed:.1f}s)")
print(json.dumps(d, indent=2))
print(f"\nStage : {d.get('state')}")
print(f"Reply : {d.get('reply')}")

if d.get("brief"):
    print("\n📋 CLINICAL BRIEF:")
    print(json.dumps(d["brief"], indent=2))

FALLBACK = {"Could you tell me more?", "", None, "Could you please repeat that?"}
if d.get("reply") in FALLBACK:
    print("\n⚠️  FALLBACK reply — dumping api.log:")
    print(open("/content/api.log").read()[-2000:])

# Quick GPU check
smi = subprocess.run(["nvidia-smi","--query-gpu=memory.used,memory.total",
                      "--format=csv,noheader"], capture_output=True, text=True)
ps  = subprocess.run(["ollama","ps"], capture_output=True, text=True)
print(f"\nGPU RAM  : {smi.stdout.strip()}")
print(f"ollama ps: {ps.stdout.strip()}")
print(subprocess.run(["tail","-n","15","/content/api.log"],
                     capture_output=True, text=True).stdout)

Session: debug_session_001  |  Message: I have chest pain

HTTP 200  (6.2s)
{
  "reply": "Can you tell me more about when this chest pain started?",
  "state": "hpi",
  "brief": null
}

Stage : hpi
Reply : Can you tell me more about when this chest pain started?

GPU RAM  : 5369 MiB, 15360 MiB
ollama ps: NAME           ID              SIZE      PROCESSOR    CONTEXT    UNTIL            
llama3.1:8b    46e0c10c039e    5.5 GB    100% GPU     4096       2 hours from now
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:7860 (Press CTRL+C to quit)
INFO:     127.0.0.1:49798 - "GET /health HTTP/1.1" 200 OK

[1777158113.511] [API] -> POST /chat received for debug_session_001
[1777158113.512] [API] Read existing state snapshot.
[1777158113.512] [API] Starting new graph invoke...
[1777158113.521] [Graph Node] Requesting LLM inference...
[Ollama] Starting inference for model 'llama3.1:8b'...
[Ollama] Inference completed 

In [ ]:
import subprocess
for log in ["/content/api.log", "/content/ollama.log"]:
    print(f"\n{'='*55}\n {log}\n{'='*55}")
    print(subprocess.run(["tail","-n","40",log], capture_output=True, text=True).stdout or "(empty)")



 /content/api.log
INFO:     Started server process [19612]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:7860 (Press CTRL+C to quit)
INFO:     127.0.0.1:49798 - "GET /health HTTP/1.1" 200 OK

[1777158113.511] [API] -> POST /chat received for debug_session_001
[1777158113.512] [API] Read existing state snapshot.
[1777158113.512] [API] Starting new graph invoke...
[1777158113.521] [Graph Node] Requesting LLM inference...
[Ollama] Starting inference for model 'llama3.1:8b'...
[Ollama] Inference completed in 6.17s total.
[1777158119.696] [Graph Node] LLM returned. Preparing node dictionaries...
[1777158119.697] [API] <- Graph invoke returned in 6.19s
[1777158119.698] [API] Chat completed in 6.19s total. Reply length: 56
INFO:     127.0.0.1:49810 - "POST /chat HTTP/1.1" 200 OK


 /content/ollama.log
load_tensors:   CPU_Mapped model buffer size =   281.81 MiB
load_tensors:        CUDA0 model buffer size =  4403

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

In [ ]:
import subprocess, os, time

subprocess.run(["pkill","-f","ollama"])
time.sleep(3)

env = os.environ.copy()
env.pop("CUDA_VISIBLE_DEVICES", None)

subprocess.Popen(["ollama","serve"], env=env)
print("Restarted Ollama")

Restarted Ollama
